# 05.03_OrthoFinder_results_processing_Python

整理/绘制 OrthoFinder 结果。

- 当前文件：`analysis/05_genome_analysis/05.03_OrthoFinder_results_processing_Python.ipynb`
- 原始来源：`Codes/05.03_OrthoFinder_results_processing.ipynb`（旧编号仅用于溯源）。
- 运行内核：**python**。
- 导入依赖：`matplotlib.patheffects`, `matplotlib.pyplot`, `pandas`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


## 1.基因复制事件

In [ ]:
import pandas as pd

In [ ]:
dup_all = pd.read_csv('/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_input/OrthoFinder/Results_Jul21/Gene_Duplication_Events/Duplications.tsv', sep='\t')
dup_all

In [ ]:
dup_all.columns

In [ ]:
dup_all['Species Tree Node'].value_counts()

In [ ]:
dup_N1 = dup_all[(dup_all['Species Tree Node'] == 'N1') & (dup_all['Support'] >= 0.5)].copy()
dup_N1

In [ ]:
dup_N2 = dup_all[(dup_all['Species Tree Node'] == 'N2') & (dup_all['Support'] >= 0.5)].copy()
dup_N2

## 2.OrthoFinder 结果可视化

### OGs Statistics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects

# 读取 Statistics_PerSpecies.tsv
file_path = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step1_QualityControl/Statistics_PerSpecies.tsv"
df = pd.read_csv(file_path, sep="\t")

# 提取三类数据
protein = df[df["Unnamed: 0"] == "Number of genes"].iloc[0, 1:].astype(int)
cluster = df[df["Unnamed: 0"] == "Number of orthogroups containing species"].iloc[0, 1:].astype(int)
singletons = df[df["Unnamed: 0"] == "Number of unassigned genes"].iloc[0, 1:].astype(int)

# 物种顺序
species_order = ["Spla.protein", "ClH23.protein", "HoH13.protein", "TrH2.protein",
                 "TrH1.protein", "Clhe.protein", "Auco.protein", "Neve.protein", "Dare.protein"]

protein = protein[species_order]
cluster = cluster[species_order]
singletons = singletons[species_order]

# 绘制横向堆叠柱状图
fig, ax = plt.subplots(figsize=(12, 7))

bar1 = ax.barh(species_order, cluster, label="cluster", color="#4C9AFF")
bar2 = ax.barh(species_order, protein - cluster - singletons, left=cluster, label="protein", color="#36B37E")
bar3 = ax.barh(species_order, singletons, left=protein - singletons, label="singletons", color="#172B4D")

# 添加数值标签（带白边）
for bars in [bar1, bar2, bar3]:
    for rect in bars:
        width = rect.get_width()
        if width > 0:
            text = ax.text(rect.get_x() + width/2, rect.get_y() + rect.get_height()/2,
                           f"{int(width)}", ha="center", va="center", fontsize=8, color="white")
            text.set_path_effects([path_effects.Stroke(linewidth=1.5, foreground="black"),
                                   path_effects.Normal()])

ax.set_xlabel("Gene / Orthogroup Count")
ax.set_ylabel("Species")
ax.set_title("Per-species statistics (Cluster / Protein / Singletons)")
ax.legend()

plt.tight_layout()
plt.show()


### Upset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 读取 Orthogroups.GeneCount.tsv 文件
file_path = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step3_OrthogroupsGeneCount/Orthogroups.GeneCount.tsv"
df = pd.read_csv(file_path, sep="\t")

# 去掉 Total 列（只保留物种列）
df = df.drop(columns=["Total"], errors="ignore")

# 将基因数转为 presence/absence (1/0)
binary_df = df.iloc[:, 1:].applymap(lambda x: 1 if x > 0 else 0)
species = list(binary_df.columns)

# 设置物种展示顺序
species_order = ["Spla.protein", "ClH23.protein", "HoH13.protein", "TrH2.protein", "TrH1.protein", "Clhe.protein", "Auco.protein", "Neve.protein", "Dare.protein"]
binary_df = binary_df[species_order]  # 按照顺序重新排列列

# 为每个直系同源群生成一个唯一的二进制模式
binary_df["pattern"] = binary_df.apply(lambda row: "".join(row.astype(str)), axis=1)

# 统计每种模式的数量
pattern_counts = binary_df["pattern"].value_counts()

# 只展示前20个交集
top_patterns = pattern_counts.head(20)

# 绘制 upset 风格的图
fig = plt.figure(figsize=(14, 8))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1])

# 柱状图（交集大小）
ax_bar = fig.add_subplot(gs[0])
bars = ax_bar.bar(range(len(top_patterns)), top_patterns.values)
ax_bar.set_xticks([])
ax_bar.set_ylabel("Cluster Count")
ax_bar.set_title("Top 20 Orthogroup Intersections")

# 在柱子上标注数值
for bar, value in zip(bars, top_patterns.values):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                str(value), ha='center', va='bottom', fontsize=8)

# dot matrix（物种组合）
ax_dot = fig.add_subplot(gs[1], sharex=ax_bar)
for i, pattern in enumerate(top_patterns.index):
    for j, bit in enumerate(pattern):
        if bit == "1":
            ax_dot.plot(i, j, "o", color="orange")
        else:
            ax_dot.plot(i, j, "o", color="lightgrey", alpha=0.5)

ax_dot.set_yticks(range(len(species_order)))
ax_dot.set_yticklabels(species_order)
ax_dot.set_xlabel("Intersection index")
ax_dot.set_xlim(-0.5, len(top_patterns)-0.5)

plt.tight_layout()
# plt.savefig("/mnt/data/upset_full_ordered.png", dpi=300, bbox_inches="tight")
# plt.close()
plt.show()


## Orthogroups提取 Protein-OG 映射表

In [ ]:
import pandas as pd

In [ ]:
# 读取文件
og_df = pd.read_csv('/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_input/OrthoFinder/Results_Jul21/Orthogroups/Orthogroups.tsv', sep='\t', dtype=str).fillna('')
og_df

In [ ]:
# 初始化物种 → gene:OG 映射的字典
species_prot_to_og = {}

# 提取所有物种列（除了第一列是 OG ID）
species_list = list(og_df.columns[1:])

# 遍历每个 Orthogroup
for _, row in og_df.iterrows():
    og = row['Orthogroup']
    for species in species_list:
        proteins = [p.strip() for p in row[species].split(',') if p.strip()]
        for protein in proteins:
            species_prot_to_og.setdefault(species, {})[protein] = og

In [ ]:
# species_prot_to_og['Auco.protein']
# {'XLOC-000277#XLOC-000277': 'OG0000000',
#  'XLOC-000355#XLOC-000355': 'OG0000000',
#  'XLOC-000411#XLOC-000411': 'OG0000000',
#  ...}

In [ ]:
for species, mapping in species_prot_to_og.items():
    df = pd.DataFrame(mapping.items(), columns=['protein_id', 'orthogroup'])
    df.to_csv(f'/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup/{species}_to_orthogroup.csv', index=False)